
## Folder 05 — File 06: Data Quality Dashboard:


It will provide:

- Overall quality status by dataset.
- Completeness and uniqueness metrics.
- Data type and format failures.
- Business rule failures.
- Quarantine counts.
- A consolidated dashboard-ready Delta table.

**Important**: This creates the data behind a dashboard. It does not automatically create a Databricks SQL dashboard UI.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

print("Libraries imported successfully.")

## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

SOURCE_REPORTS = {
    "data_quality": f"{CATALOG}.{SCHEMA}.data_quality_report",

    "completeness_uniqueness": (
        f"{CATALOG}.{SCHEMA}.completeness_uniqueness_report"
    ),

    "data_type_format": (
        f"{CATALOG}.{SCHEMA}.data_type_format_report"
    ),

    "business_rules": (
        f"{CATALOG}.{SCHEMA}.business_rule_validation_report"
    )
}

QUARANTINE_SUMMARY_TABLE = (
    f"{CATALOG}.{SCHEMA}.quarantine_summary"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SCHEMA}.quarantine_records"
)

DASHBOARD_DETAIL_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_detail"
)

DASHBOARD_SUMMARY_TABLE = (
    f"{CATALOG}.{SCHEMA}.data_quality_dashboard_summary"
)

print("Configuration completed.")

## 3. Define reusable report normalization function

#  Different validation notebooks may use slightly different
#  column names. Normalize them into a common structure.

def first_available_column(df, candidates, default_value=None):

    available_columns = set(df.columns)

    for candidate in candidates:

        if candidate in available_columns:

            return F.col(candidate)

    return F.lit(default_value)


def normalize_report(df, source_name):

    dataset_column = first_available_column(
        df,
        [
            "dataset_name",
            "_quarantine_dataset",
            "table_name"
        ],
        "unknown"
    )

    check_column = first_available_column(
        df,
        [
            "check_name",
            "rule_name",
            "_quarantine_reason"
        ],
        "unspecified_check"
    )

    column_column = first_available_column(
        df,
        [
            "column_name"
        ],
        None
    )

    total_column = first_available_column(
        df,
        [
            "total_records",
            "total_count"
        ],
        None
    )

    failed_column = first_available_column(
        df,
        [
            "failed_records",
            "invalid_records",
            "quarantined_records"
        ],
        0
    )

    failure_percentage_column = first_available_column(
        df,
        [
            "failure_percentage"
        ],
        None
    )

    status_column = first_available_column(
        df,
        [
            "status"
        ],
        None
    )

    details_column = first_available_column(
        df,
        [
            "details",
            "description"
        ],
        None
    )

    checked_at_column = first_available_column(
        df,
        [
            "checked_at",
            "summary_generated_at",
            "_quarantined_at"
        ],
        None
    )

    normalized_df = df.select(

        dataset_column.cast("string").alias("dataset_name"),

        F.concat(
            check_column.cast("string"),
            F.when(
                column_column.isNotNull(),
                F.concat(
                    F.lit(" - "),
                    column_column.cast("string")
                )
            ).otherwise(F.lit(""))
        ).alias("check_name"),

        F.lit(source_name).alias("source_report"),

        total_column.cast("long").alias("total_records"),

        failed_column.cast("long").alias("failed_records"),

        failure_percentage_column.cast("double").alias(
            "failure_percentage"
        ),

        status_column.cast("string").alias("source_status"),

        details_column.cast("string").alias("details"),

        checked_at_column.cast("timestamp").alias("checked_at")
    )

    normalized_df = normalized_df.withColumn(

        "status",

        F.when(
            F.col("source_status").isNotNull(),
            F.upper(F.col("source_status"))
        ).when(
            F.col("failed_records") > 0,
            F.lit("FAIL")
        ).otherwise(
            F.lit("PASS")
        )
    )

    return normalized_df.drop("source_status")


## 4. Load and normalize available validation reports

normalized_reports = []

for source_name, table_name in SOURCE_REPORTS.items():

    if spark.catalog.tableExists(table_name):

        source_df = spark.table(table_name)

        normalized_df = normalize_report(
            source_df,
            source_name
        )

        normalized_reports.append(normalized_df)

        print(f"Loaded report: {table_name}")

    else:

        print(f"WARNING: Report not found: {table_name}")

if not normalized_reports:

    raise Exception(
        "No validation reports were found. "
        "Run the previous data quality notebooks first."
    )

## 5. Add quarantine metrics

#  Quarantine summary records represent invalid-record counts.
#  They are included as separate monitoring checks.

if spark.catalog.tableExists(QUARANTINE_SUMMARY_TABLE):

    quarantine_summary_df = spark.table(
        QUARANTINE_SUMMARY_TABLE
    )

    quarantine_report_df = normalize_report(
        quarantine_summary_df,
        "quarantine_summary"
    )

    normalized_reports.append(
        quarantine_report_df
    )

    print("Quarantine summary loaded.")

else:

    print("WARNING: Quarantine summary table not found.")


## 6. Combine all normalized reports

dashboard_detail_df = normalized_reports[0]

for next_report in normalized_reports[1:]:

    dashboard_detail_df = (
        dashboard_detail_df.unionByName(
            next_report,
            allowMissingColumns=True
        )
    )

dashboard_detail_df = (
    dashboard_detail_df
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
)

display(
    dashboard_detail_df.orderBy(
        "dataset_name",
        "source_report",
        "check_name"
    )
)


## 7. Add current quarantine record counts

if spark.catalog.tableExists(QUARANTINE_TABLE):

    current_quarantine_df = spark.table(
        QUARANTINE_TABLE
    )

    quarantine_counts_df = (
        current_quarantine_df
        .groupBy("_quarantine_dataset")
        .agg(
            F.count("*").alias("current_quarantined_records")
        )
        .withColumnRenamed(
            "_quarantine_dataset",
            "dataset_name"
        )
    )

else:

    quarantine_counts_df = spark.createDataFrame(
        [],
        "dataset_name STRING, current_quarantined_records LONG"
    )

## 8. Generate dataset-level dashboard summary

#  Check pass rate is the percentage of configured checks passing.
#  It is not a percentage of accurate source records.

dataset_summary_df = (
    dashboard_detail_df
    .groupBy("dataset_name")
    .agg(

        F.count("*").alias("total_checks"),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        ).alias("failed_checks"),

        F.sum(
            F.coalesce(
                F.col("failed_records"),
                F.lit(0)
            )
        ).alias("reported_failed_records"),

        F.max("checked_at").alias("last_checked_at")
    )
    .withColumn(
        "check_pass_rate",
        F.round(
            F.col("passed_checks") /
            F.col("total_checks") * 100,
            2
        )
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_checks") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .join(
        quarantine_counts_df,
        on="dataset_name",
        how="left"
    )
    .withColumn(
        "current_quarantined_records",
        F.coalesce(
            F.col("current_quarantined_records"),
            F.lit(0)
        )
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
    .orderBy("dataset_name")
)

display(dataset_summary_df)


## 9. Generate overall dashboard KPIs

overall_summary_df = (
    dashboard_detail_df
    .agg(

        F.count("*").alias("total_checks"),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        ).alias("passed_checks"),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        ).alias("failed_checks"),

        F.sum(
            F.coalesce(
                F.col("failed_records"),
                F.lit(0)
            )
        ).alias("reported_failed_records")
    )
    .withColumn(
        "check_pass_rate",
        F.round(
            F.col("passed_checks") /
            F.col("total_checks") * 100,
            2
        )
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_checks") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
    .withColumn(
        "dashboard_generated_at",
        F.current_timestamp()
    )
)

display(overall_summary_df)


## 10. Generate failure breakdown by report type

failure_breakdown_df = (
    dashboard_detail_df
    .groupBy("source_report", "status")
    .agg(
        F.count("*").alias("number_of_checks"),

        F.sum(
            F.coalesce(
                F.col("failed_records"),
                F.lit(0)
            )
        ).alias("reported_failed_records")
    )
    .orderBy(
        "source_report",
        "status"
    )
)

display(failure_breakdown_df)


## 11. Generate most common failure checks

top_failures_df = (
    dashboard_detail_df
    .filter(
        F.col("status") == "FAIL"
    )
    .groupBy(
        "dataset_name",
        "check_name",
        "source_report"
    )
    .agg(
        F.sum(
            F.coalesce(
                F.col("failed_records"),
                F.lit(0)
            )
        ).alias("reported_failed_records"),

        F.count("*").alias("failed_check_occurrences")
    )
    .orderBy(
        F.desc("reported_failed_records"),
        F.desc("failed_check_occurrences")
    )
)

display(top_failures_df)


## 12. Save dashboard detail table

(
    dashboard_detail_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_DETAIL_TABLE)
)

print(
    f"Dashboard detail saved: {DASHBOARD_DETAIL_TABLE}"
)


## 13. Save dashboard summary table

(
    dataset_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DASHBOARD_SUMMARY_TABLE)
)

print(
    f"Dashboard summary saved: {DASHBOARD_SUMMARY_TABLE}"
)


## 14. Final dashboard output

print("=" * 65)
print("E-COMMERCE DATA QUALITY DASHBOARD")
print("=" * 65)

display(
    spark.table(DASHBOARD_SUMMARY_TABLE)
)

display(
    spark.table(DASHBOARD_DETAIL_TABLE)
)

print("=" * 65)
print("DATA QUALITY DASHBOARD PREPARATION COMPLETED")
print("=" * 65)

print(f"Detail table  : {DASHBOARD_DETAIL_TABLE}")
print(f"Summary table : {DASHBOARD_SUMMARY_TABLE}")
